In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os
import pyBigWig
import seaborn as sns
import matplotlib.pyplot as plt
import h5py
import numpy as np
import pandas as pd

In [ ]:
in_dir = "../../results/03_accessbpnet/16_variant_score_prediction"
out_dir = "../../results/03_accessbpnet/19_variant_effects_to_bw"
os.makedirs(out_dir, exist_ok=True)

chrom_sizes_file = "../../data/GRCh38/GRCh38.primary_assembly.chrom.sizes"

In [ ]:
chrom_sizes = pd.read_csv(chrom_sizes_file, header=None, sep='\t', names=['chrom', 'size'])
chrom_sizes_dict = chrom_sizes.set_index('chrom')['size'].to_dict()

In [ ]:
df_atac = pd.read_csv(f"../../results/03_accessbpnet/17_eval_variant_score/ATAC.variant_scores.merged.tsv", sep='\t')
df_access = pd.read_csv(f"../../results/03_accessbpnet/17_eval_variant_score/ACCESS.variant_scores.merged.tsv", sep='\t')

df_atac = df_atac[["chr", "pos", "allele1", "allele2", "logfc", "beta"]]
df_access = df_access[["chr", "pos", "allele1", "allele2", "logfc"]]

df_atac.rename(columns={"logfc": "logfc_atac"}, inplace=True)
df_access.rename(columns={"logfc": "logfc_access"}, inplace=True)

df = pd.merge(df_atac, df_access, on=["chr", "pos", "allele1", "allele2"], how="outer")
df = df[abs(df['beta']) > 1.5].reset_index(drop=True)

In [ ]:
df["beta"].hist()

In [ ]:
df_snp = pd.read_csv("../../results/03_accessbpnet/15_prepare_snp/snp_variants.tsv", sep='\t', header=None)
df_snp.columns = ['chr', 'pos', 'allele1', 'allele2', 'id']

In [ ]:
df_snp

In [ ]:
# get index for selected variants
df_snp = pd.merge(df, df_snp, on=['chr', 'pos', 'allele1', 'allele2'], how='right')

In [ ]:
df_snp.dropna(inplace=True)

In [ ]:
df_snp.sort_values(by=['beta'], inplace=True, ascending=False)

In [ ]:
df_snp

In [ ]:
# df_snp = df_snp.head(1)

In [ ]:
# --- Load predicted profiles for both assays once ---
assays = ['ATAC', 'ACCESS']
assay_colors = {'ATAC': 'green', 'ACCESS': 'purple'}

profiles = {}   # profiles[assay] = (allele1_pred_profiles, allele2_pred_profiles)
h5_handles = {}
for assay in assays:
    f = h5py.File(
        f"../../results/03_accessbpnet/16_variant_score_prediction/{assay}.variant_predictions.h5", "r"
    )
    h5_handles[assay] = f
    profiles[assay] = (
        f['observed']['allele1_pred_profiles'],
        f['observed']['allele2_pred_profiles'],
    )

# --- Plot: one figure per variant, both assays overlaid ---
for idx, row in df_snp.iterrows():
    chrom = row['chr']
    pos = row['pos']
    beta = row['beta']
    allele1 = row['allele1']
    allele2 = row['allele2']

    logfc_atac = row['logfc_atac']
    logfc_access = row['logfc_access']

    variant_id = row['id'].replace(":", "_")

    # rows: ATAC, ACCESS; cols: full view, zoomed view
    fig, axes = plt.subplots(len(assays), 2, figsize=(14, 4 * len(assays)))

    for a, assay in enumerate(assays):
        allele1_pred_profiles, allele2_pred_profiles = profiles[assay]

        allele1_scores = allele1_pred_profiles[idx, :].astype(np.float32)
        allele2_scores = allele2_pred_profiles[idx, :].astype(np.float32)

        # symmetrize (fwd + rev-complement strand)
        allele1_scores = allele1_scores + allele1_scores[::-1]
        allele2_scores = allele2_scores + allele2_scores[::-1]

        logfc = logfc_atac if assay == 'ATAC' else logfc_access

        # --- full view ---
        ax = axes[0, a]
        ax.plot(allele1_scores, label=f'Allele 1: {allele1}', color='blue')
        ax.plot(allele2_scores, label=f'Allele 2: {allele2}', color='orange')
        ax.axvline(x=501, color='red', linestyle='--', label='Variant position')
        ax.set_title(f'{assay} — {variant_id} (beta: {beta:.3f}; logfc: {logfc:.3f})')
        ax.set_xlabel('Position (relative to variant)')
        ax.set_ylabel('Predicted score')
        ax.set_xlim(0, 1001)
        ax.set_xticks([0, 250, 500, 750, 1000])
        ax.set_xticklabels([-500, -250, 0, 250, 500])
        ax.legend(fontsize=8)

        # --- zoomed view (+/- 50 bp) ---
        zoom_start, zoom_end = 501 - 50, 501 + 50
        ax = axes[1, a]
        ax.plot(allele1_scores[zoom_start:zoom_end], label=f'Allele 1: {allele1}', color='blue')
        ax.plot(allele2_scores[zoom_start:zoom_end], label=f'Allele 2: {allele2}', color='orange')
        ax.axvline(x=50, color='red', linestyle='--', label='Variant position')
        ax.set_title(f'{assay} — zoom (+/- 50 bp)')
        ax.set_xlabel('Position (relative to variant)')
        ax.set_ylabel('Predicted score')
        ax.set_xlim(0, 101)
        ax.set_xticks([0, 25, 50, 75, 100])
        ax.set_xticklabels([-50, -25, 0, 25, 50])
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(f"{out_dir}/{variant_id}_predicted_profiles.png", dpi=150)
    plt.close()

    # --- combined TSV: long format with assay column ---
    dfs = []
    for assay in assays:
        allele1_pred_profiles, allele2_pred_profiles = profiles[assay]
        a1 = allele1_pred_profiles[idx, :].astype(np.float32)
        a2 = allele2_pred_profiles[idx, :].astype(np.float32)
        a1 = a1 + a1[::-1]
        a2 = a2 + a2[::-1]
        d = pd.DataFrame({
            'position': np.arange(-500, 500),
            'allele1_score': a1,
            'allele2_score': a2,
        })
        d = pd.melt(d, id_vars=['position'], value_vars=['allele1_score', 'allele2_score'],
                    var_name='allele', value_name='predicted_score')
        d['assay'] = assay
        dfs.append(d)
    pd.concat(dfs, ignore_index=True).to_csv(
        f"{out_dir}/{variant_id}_predicted_profiles.tsv", sep='\t', index=False
    )

# close all h5 handles
for f in h5_handles.values():
    f.close()